## Exported by node, edge file for each group networks.

## Import RDS file

In [1]:
library(SpiecEasi)
library(phyloseq)
library(NetCoMi)
library(igraph)
library(dplyr)




Attaching package: ‘igraph’


The following object is masked from ‘package:SpiecEasi’:

    make_graph


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union



Attaching package: ‘dplyr’


The following objects are masked from ‘package:igraph’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:

## high, networks_pos is clustered only used positive correlation.
high_network <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/high_network_pos.rds")
high_genus_renamed <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/high_phyloseq.rds")
high_genus <- high_genus_renamed$high_genus_renamed ### phyloseq allocation by function
taxtab_high <- high_genus_renamed$taxtab_high
high_net_genus <- high_network$net_genus
high_props_genus <- high_network$props_genus

## low
low_network <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/low_network_pos.rds")
low_genus_renamed <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/low_phyloseq.rds")
low_genus <- low_genus_renamed$low_genus_renamed ### phyloseq allocation by function
taxtab_low <- low_genus_renamed$taxtab_low
low_net_genus <- low_network$net_genus
low_props_genus <- low_network$props_genus

## pibd
pibd_network <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/pibd_network_pos.rds")
pibd_genus_renamed <- readRDS("data/3_results/5_network_analysis/5_1_network_inference/pibd_phyloseq.rds")
pibd_genus <- pibd_genus_renamed$pibd_genus_renamed ### phyloseq allocation by function
taxtab_pibd <- pibd_genus_renamed$taxtab_pibd
pibd_net_genus <- pibd_network$net_genus
pibd_props_genus <- pibd_network$props_genus


## MMI high -> MMI low -> PIBD

In [ ]:
## High version
graph3 <- igraph::graph_from_adjacency_matrix(high_net_genus$adjaMat1, 
                                              weighted = TRUE)
set.seed(42) 
ps_lay_fr <- igraph::layout_with_fr(graph3)


#5. Over edge weighted > 0.4 filtering by csv export method 
edges_raw <- dplyr::filter(high_net_genus$edgelist1)

## First of all, Choose by all nodes
all_nodes <- unique(c(edges_raw$v1, edges_raw$v2))
nodes <- data.frame(Label = all_nodes, stringsAsFactors = FALSE)
nodes$Id <- seq_len(nrow(nodes))  

  
 ## High node representitative nodes number dataframing.
  nodes_repre <- data.frame(Label = all_nodes, stringsAsFactors = FALSE)
  nodes_repre$Id <- seq_len(nrow(nodes)) 

## Second, Label → ID mapping in edges
node_map <- setNames(nodes$Id, nodes$Label)
edges <- edges_raw %>%
  mutate(Source = edges_raw$v1,
         Target = edges_raw$v2,
         Type = "Undirected",
         Weight = adja,
         Sign = ifelse(asso > 0, "Positive", "Negative")) %>%
  select(Source, Target, Type, Weight, Sign, asso)

## Third, Add to information(cluster number)
nodes$cluster <- high_props_genus$clustering$clust1[nodes$Label]
nodes$clr <- high_props_genus$nodeAttr$clr[nodes$Label]
nodes$degree <- high_props_genus$centralities$degree1[nodes$Label] # Degree
nodes$betweenness <- high_props_genus$centralities$between1[nodes$Label] # Betweenness centrality
nodes$closeness <- high_props_genus$centralities$close1[nodes$Label] # Closeness centrality
nodes$eigenvector <- high_props_genus$centralities$eigenv1[nodes$Label] # Eigenvector centrality

## fourth, Add to prevalence, abundance, phylum infomation by each node
woo_phyla <- as.factor(taxtab_high[, "Phylum"])
total_reads <- taxa_sums(high_genus)

woo_df <- data.frame(
  Label = taxa_names(high_genus),
  abundance = as.numeric(total_reads),
  phylum = as.character(woo_phyla),
  stringsAsFactors = FALSE
)

nodes <- merge(nodes, woo_df, by = "Label", all.x = TRUE)

#6. Export to csv file(nodes, edges)
nodes <- nodes[order(nodes$Id), ]
write.csv(nodes, "data/3_results/5_network_analysis/5_2_network_export/high/high_nodes_topology.csv", row.names = FALSE, quote = FALSE)
write.csv(edges, "data/3_results/5_network_analysis/5_2_network_export/high/high_edges.csv", row.names = FALSE, quote = FALSE)






In [ ]:
## low version

graph3 <- igraph::graph_from_adjacency_matrix(low_net_genus$adjaMat1, 
                                              weighted = TRUE)
set.seed(42) 
ps_lay_fr <- igraph::layout_with_fr(graph3)


#5. Over edge weighted > 0.4 filtering by csv export method 
edges_raw <- dplyr::filter(low_net_genus$edgelist1)

## First of all, Choose by all nodes
all_nodes <- unique(c(edges_raw$v1, edges_raw$v2))
nodes <- data.frame(Label = all_nodes, stringsAsFactors = FALSE)
nodes$Id <- seq_len(nrow(nodes)) 


## Second, Label → ID mapping in edges
node_map <- setNames(nodes$Id, nodes$Label)
edges <- edges_raw %>%
  mutate(Source = edges_raw$v1,
         Target = edges_raw$v2,
         Type = "Undirected",
         Weight = adja,
         Sign = ifelse(asso > 0, "Positive", "Negative")) %>%
  select(Source, Target, Type, Weight, Sign, asso)

## Third, Add to information(cluster number)
nodes$cluster <- low_props_genus$clustering$clust1[nodes$Label]
nodes$clr <- low_props_genus$nodeAttr$clr[nodes$Label]
nodes$degree <- low_props_genus$centralities$degree1[nodes$Label] # Degree
nodes$betweenness <- low_props_genus$centralities$between1[nodes$Label] # Betweenness centrality
nodes$closeness <- low_props_genus$centralities$close1[nodes$Label] # Closeness centrality
nodes$eigenvector <- low_props_genus$centralities$eigenv1[nodes$Label] # Eigenvector centrality

## fourth, Add to prevalence, abundance, phylum infomation by each node
woo_phyla <- as.factor(taxtab_low[, "Phylum"])
total_reads <- taxa_sums(low_genus)

woo_df <- data.frame(
  Label = taxa_names(low_genus),
  abundance = as.numeric(total_reads),
  phylum = as.character(woo_phyla),
  stringsAsFactors = FALSE
)

nodes <- merge(nodes, woo_df, by = "Label", all.x = TRUE)


#6. Export to csv file(nodes, edges)
nodes <- nodes[order(nodes$Id), ]
write.csv(nodes, "data/3_results/5_network_analysis/5_2_network_export/low/low_nodes_topology.csv", row.names = FALSE)
write.csv(edges, "data/3_results/5_network_analysis/5_2_network_export/low/low_edges.csv", row.names = FALSE)





In [8]:
nodes

,Label,Id,cluster,degree,betweenness,closeness,eigenvector,abundance,phylum
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
41,Escherichia,1,2,0.031914894,0.005602241,0.5893803,3.419880e-01,16109,Pseudomonadota
54,Gemmiger_A_73129,2,1,0.074468085,0.206162465,0.7764885,7.201189e-02,13408,Bacillota_A_368345
72,Methanobrevibacter_A,3,4,0.005319149,0.000000000,0.3965703,3.409790e-04,1113,Methanobacteriota_A_1229
18,Bacteroides_H_857956,4,3,0.021276596,0.000000000,0.5810092,1.097175e-02,12285,Bacteroidota
21,Bifidobacterium_388775,5,2,0.037234043,0.049579832,0.6936020,4.346413e-01,33102,Actinomycetota
40,Enterococcus_B,6,2,0.005319149,0.000000000,0.4643064,4.014577e-02,3531,Bacillota_I
22,Blautia_A_141780,7,2,0.047872340,0.011764706,0.6479538,6.125070e-01,809,Bacillota_A_368345
43,Eubacterium_G_180878,8,1,0.015957447,0.000000000,0.5562608,1.297814e-02,243,Bacillota_A_368345
7,1_Lachnospiraceae(F),9,2,0.021276596,0.082633053,0.6037520,2.411120e-01,534,Bacillota_A_368345


In [ ]:
## pibd


graph3 <- igraph::graph_from_adjacency_matrix(pibd_net_genus$adjaMat1, 
                                              weighted = TRUE)
set.seed(42) 
ps_lay_fr <- igraph::layout_with_fr(graph3)


#5. Over edge weighted > 0.4 filtering by csv export method 
edges_raw <- dplyr::filter(pibd_net_genus$edgelist1)

## First of all, Choose by all nodes
all_nodes <- unique(c(edges_raw$v1, edges_raw$v2))
nodes <- data.frame(Label = all_nodes, stringsAsFactors = FALSE)
nodes$Id <- seq_len(nrow(nodes))  


## Second, Label → ID mapping in edges
node_map <- setNames(nodes$Id, nodes$Label)
edges <- edges_raw %>%
  mutate(Source = edges_raw$v1,
         Target = edges_raw$v2,
         Type = "Undirected",
         Weight = adja,
         Sign = ifelse(asso > 0, "Positive", "Negative")) %>%
  select(Source, Target, Type, Weight, Sign, asso)

## Third, Add to information(cluster number)
nodes$cluster <- pibd_props_genus$clustering$clust1[nodes$Label]
nodes$clr <- pibd_props_genus$nodeAttr$clr[nodes$Label]
nodes$degree <- pibd_props_genus$centralities$degree1[nodes$Label] # Degree
nodes$betweenness <- pibd_props_genus$centralities$between1[nodes$Label] # Betweenness centrality
nodes$closeness <- pibd_props_genus$centralities$close1[nodes$Label] # Closeness centrality
nodes$eigenvector <- pibd_props_genus$centralities$eigenv1[nodes$Label] # Eigenvector centrality

## fourth, Add to prevalence, abundance, phylum infomation by each node
woo_phyla <- as.factor(taxtab_pibd[, "Phylum"])
total_reads <- taxa_sums(pibd_genus)

woo_df <- data.frame(
  Label = taxa_names(pibd_genus),
  abundance = as.numeric(total_reads),
  phylum = as.character(woo_phyla),
  stringsAsFactors = FALSE
)

nodes <- merge(nodes, woo_df, by = "Label", all.x = TRUE)

#6. Export to csv file(nodes, edges)
nodes <- nodes[order(nodes$Id), ]
write.csv(nodes, "data/3_results/5_network_analysis/5_2_network_export/pibd/pibd_nodes_topology.csv", row.names = FALSE)
write.csv(edges, "data/3_results/5_network_analysis/5_2_network_export/pibd/pibd_edges.csv", row.names = FALSE)





In [11]:
nodes

,Label,Id,cluster,degree,betweenness,closeness,eigenvector,abundance,phylum
,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1_Bacteria(K),1,1,0.010638298,0.000617284,0.3657093,0.0003146949,210,__
33,Escherichia,2,1,0.005319149,0.000000000,0.3457030,0.0002794848,30130,Pseudomonadota
44,Gemmiger_A_73129,3,2,0.026595745,0.002160494,0.5524187,0.4106847737,10823,Bacillota_A_368345
14,Bacteroides_H_857956,4,2,0.037234043,0.048765432,0.5989270,0.5805189058,27198,Bacteroidota
52,Hydrotalea,5,11,0.000000000,0.000000000,0.0000000,0.0000000000,126,Bacteroidota
16,Bifidobacterium_388775,6,3,0.010638298,0.000308642,0.4894817,0.0121147770,21583,Actinomycetota
31,Enterococcus_B,7,6,0.015957447,0.054012346,0.5249255,0.0049287654,4619,Bacillota_I
17,Blautia_A_141780,8,3,0.042553191,0.083333333,0.6433284,0.0491272417,1450,Bacillota_A_368345
34,Eubacterium_G_180878,9,2,0.010638298,0.000000000,0.4996823,0.1817755002,625,Bacillota_A_368345
